# POS Tagging Model Evaluation

This notebook:
1. Loads a trained POS tagging model for a specific language
2. Displays training curves per epoch
3. Tests the model on custom sentences

## 1. Configuration and Setup

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import json
from pathlib import Path

sys.path.append(os.path.join(os.getcwd(), 'src'))

from model import create_model
from data import load_data_and_dataloaders
import config

In [ ]:
# PARAMETRE: Choisir la langue
language = 'en'  # Changez ici: 'en', 'fr', 'de', 'es', etc.

load_dotenv()
DATA_PATH = os.getenv("UD_DATA_PATH")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Language: {config.LANGUAGES[language]['name']}")
print(f"Device: {device}")
print(f"Data path: {DATA_PATH}")

## 2. Load Training Metrics

In [ ]:
# Trouver le dernier run d'entraînement
results_dir = Path("results")
run_dirs = sorted([d for d in results_dir.iterdir() if d.is_dir()], reverse=True)

if not run_dirs:
    print("No training runs found!")
else:
    latest_run = run_dirs[0]
    print(f"Using training run: {latest_run.name}")
    
    metrics_file = latest_run / "metrics" / f"{language}_metrics.json"
    
    if metrics_file.exists():
        with open(metrics_file, 'r') as f:
            training_data = json.load(f)
        
        print(f"\nMetrics loaded successfully!")
        print(f"Best epoch: {training_data['best_epoch']}")
        print(f"Best dev F1: {training_data['best_dev_f1']:.4f}")
        print(f"Test F1: {training_data['test_metrics']['test_f1']:.4f}")
    else:
        print(f"Metrics file not found: {metrics_file}")

## 3. Display Training Curves

In [ ]:
history = training_data['training_history']
epochs = history['epochs']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training Loss
axes[0, 0].plot(epochs, history['train_loss'], marker='o', linewidth=2, markersize=6, color='#2E86AB')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(x=training_data['best_epoch'], color='red', linestyle='--', alpha=0.7, label=f"Best epoch: {training_data['best_epoch']}")
axes[0, 0].legend()

# Dev F1 Score
axes[0, 1].plot(epochs, history['dev_f1'], marker='o', linewidth=2, markersize=6, color='#06A77D')
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('F1 Score', fontsize=12)
axes[0, 1].set_title('Development F1 Score', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(x=training_data['best_epoch'], color='red', linestyle='--', alpha=0.7, label=f"Best: {training_data['best_dev_f1']:.4f}")
axes[0, 1].axhline(y=training_data['best_dev_f1'], color='green', linestyle='--', alpha=0.5)
axes[0, 1].legend()

# Dev Accuracy
axes[1, 0].plot(epochs, history['dev_accuracy'], marker='o', linewidth=2, markersize=6, color='#F18F01')
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Accuracy', fontsize=12)
axes[1, 0].set_title('Development Accuracy', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(x=training_data['best_epoch'], color='red', linestyle='--', alpha=0.7)

# Dev Precision & Recall
axes[1, 1].plot(epochs, history['dev_precision'], marker='o', linewidth=2, markersize=6, color='#C73E1D', label='Precision')
axes[1, 1].plot(epochs, history['dev_recall'], marker='s', linewidth=2, markersize=6, color='#6A4C93', label='Recall')
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Score', fontsize=12)
axes[1, 1].set_title('Development Precision & Recall', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axvline(x=training_data['best_epoch'], color='red', linestyle='--', alpha=0.7)
axes[1, 1].legend()

plt.suptitle(f'Training Curves - {config.LANGUAGES[language]["name"]}', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 4. Test Metrics Summary

In [ ]:
test_metrics = training_data['test_metrics']

print("="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Accuracy:  {test_metrics['test_accuracy']:.4f}")
print(f"Precision: {test_metrics['test_precision']:.4f}")
print(f"Recall:    {test_metrics['test_recall']:.4f}")
print(f"F1 Score:  {test_metrics['test_f1']:.4f}")
print("="*60)

## 5. Per-Tag Performance

In [ ]:
per_tag_f1 = {k: v for k, v in test_metrics['per_tag_f1'].items() if k != '<PAD>'}
tags = list(per_tag_f1.keys())
f1_scores = list(per_tag_f1.values())

plt.figure(figsize=(14, 6))
bars = plt.bar(range(len(tags)), f1_scores, color='steelblue', edgecolor='navy', linewidth=1.2)

# Colorer différemment les tags avec F1 < 0.8
for i, (tag, score) in enumerate(zip(tags, f1_scores)):
    if score < 0.8:
        bars[i].set_color('coral')

plt.xlabel('POS Tags', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.title(f'Per-Tag F1 Scores - {config.LANGUAGES[language]["name"]}', fontsize=14, fontweight='bold')
plt.xticks(range(len(tags)), tags, rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.axhline(y=test_metrics['test_f1'], color='red', linestyle='--', linewidth=2, label=f"Overall F1: {test_metrics['test_f1']:.4f}")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nTop 5 best performing tags:")
for tag, f1 in sorted(per_tag_f1.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {tag:12s}: {f1:.4f}")

print("\nTop 5 worst performing tags:")
for tag, f1 in sorted(per_tag_f1.items(), key=lambda x: x[1])[:5]:
    print(f"  {tag:12s}: {f1:.4f}")

## 6. Load Model and Data

In [ ]:
model_path = os.path.join(config.MODELS_DIR, f"best_model_{language}.pt")

print("Loading data...")
train_loader, dev_loader, test_loader, word2idx, tag2idx, char2idx = load_data_and_dataloaders(
    DATA_PATH, language=language, batch_size=config.BATCH_SIZE, use_chars=True
)

idx2tag = {v: k for k, v in tag2idx.items()}
idx2word = {v: k for k, v in word2idx.items()}

print(f"\nVocabulary size: {len(word2idx)}")
print(f"Number of tags: {len(tag2idx)}")
print(f"Tags: {list(tag2idx.keys())}")

print("\nLoading model...")
vocab_size = len(word2idx)
char_vocab_size = len(char2idx) if char2idx else 100
num_tags = len(tag2idx)

model = create_model(vocab_size, char_vocab_size, num_tags, device)

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"Model loaded successfully from {model_path}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
else:
    print(f"ERROR: Model file not found at {model_path}")

model.eval()
print("\nModel ready for inference!")

## 7. Prediction Function

In [ ]:
def predict_sentence(model, sentence, word2idx, idx2tag, char2idx, device, use_crf=True):
    """
    Predict POS tags for a single sentence.
    """
    model.eval()
    
    words = sentence.split()
    word_ids = [word2idx.get(word.lower(), word2idx.get('<UNK>', 1)) for word in words]
    
    char_ids = []
    if char2idx:
        for word in words:
            char_ids.append([char2idx.get(c, char2idx.get('<UNK>', 1)) for c in word[:config.MAX_WORD_LENGTH]])
        max_word_len = max(len(cw) for cw in char_ids) if char_ids else 0
        if max_word_len > 0:
            char_ids = [cw + [0] * (max_word_len - len(cw)) for cw in char_ids]
    
    word_tensor = torch.tensor([word_ids], dtype=torch.long).to(device)
    char_tensor = torch.tensor([char_ids], dtype=torch.long).to(device) if char_ids else None
    mask = torch.ones_like(word_tensor, dtype=torch.bool).to(device)
    
    with torch.no_grad():
        if use_crf:
            predictions = model(word_tensor, char_tensor, mask)
            pred_tags = predictions[0]
        else:
            emissions = model(word_tensor, char_tensor, mask)
            pred_tags = emissions.argmax(dim=-1)[0].cpu().numpy()
    
    predicted_tags = [idx2tag[tag_id] for tag_id in pred_tags]
    
    return list(zip(words, predicted_tags))


def display_predictions(sentence, predictions):
    """
    Display predictions in a nice format.
    """
    print(f"\nSentence: {sentence}")
    print("-" * 70)
    print(f"{'Word':<20} {'POS Tag':<15}")
    print("-" * 70)
    for word, tag in predictions:
        print(f"{word:<20} {tag:<15}")
    print("=" * 70)

## 8. Test on Custom Sentences

Définissez vos propres phrases ici pour tester le modèle!

In [ ]:
# Définir les phrases de test selon la langue
test_sentences = {
    'en': [
        "The quick brown fox jumps over the lazy dog",
        "She sells seashells by the seashore",
        "I am learning natural language processing with deep learning",
        "The cat sat on the mat",
        "John visited Paris last summer and loved it"
    ],
    'fr': [
        "Le chat noir dort sur le canapé",
        "Marie mange une pomme rouge",
        "Les enfants jouent dans le jardin",
        "Je vais à Paris demain",
        "Il fait beau aujourd'hui"
    ],
    'de': [
        "Der Hund läuft im Park",
        "Ich esse gerne Schokolade",
        "Die Kinder spielen Fußball",
        "Wir gehen morgen ins Kino",
        "Das Wetter ist schön heute"
    ],
    'es': [
        "El gato negro duerme en el sofá",
        "María come una manzana roja",
        "Los niños juegan en el jardín",
        "Voy a Madrid mañana",
        "Hace buen tiempo hoy"
    ]
}

# Sélectionner les phrases selon la langue
sentences = test_sentences.get(language, test_sentences['en'])

print("="*70)
print(f"TESTING MODEL ON CUSTOM SENTENCES - {config.LANGUAGES[language]['name'].upper()}")
print("="*70)

for sentence in sentences:
    predictions = predict_sentence(model, sentence, word2idx, idx2tag, char2idx, device, use_crf=config.USE_CRF)
    display_predictions(sentence, predictions)

## 9. Interactive Testing

Testez vos propres phrases interactivement!

In [ ]:
# Testez votre propre phrase ici
custom_sentence = "The intelligent student quickly solved the difficult problem"

predictions = predict_sentence(model, custom_sentence, word2idx, idx2tag, char2idx, device, use_crf=config.USE_CRF)
display_predictions(custom_sentence, predictions)

In [ ]:
# Boucle interactive (optionnel - décommentez pour utiliser)
def interactive_prediction():
    """
    Interactive loop for testing custom sentences.
    """
    print("Enter a sentence to tag (or 'quit' to exit):")
    
    while True:
        sentence = input("\n> ")
        
        if sentence.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        if not sentence.strip():
            continue
        
        predictions = predict_sentence(model, sentence, word2idx, idx2tag, char2idx, device, use_crf=config.USE_CRF)
        display_predictions(sentence, predictions)

# Décommentez la ligne suivante pour lancer le mode interactif
# interactive_prediction()

## 10. Analysis Summary

In [ ]:
print("="*70)
print("ANALYSIS SUMMARY")
print("="*70)
print(f"Language: {config.LANGUAGES[language]['name']}")
print(f"Model: {model_path}")
print(f"\nTraining:")
print(f"  Total epochs: {len(history['epochs'])}")
print(f"  Best epoch: {training_data['best_epoch']}")
print(f"  Best dev F1: {training_data['best_dev_f1']:.4f}")
print(f"\nTest Performance:")
print(f"  Accuracy:  {test_metrics['test_accuracy']:.4f}")
print(f"  Precision: {test_metrics['test_precision']:.4f}")
print(f"  Recall:    {test_metrics['test_recall']:.4f}")
print(f"  F1 Score:  {test_metrics['test_f1']:.4f}")
print("="*70)